# 15.071 — Deliverable 1
## Problem 4: Logistic Regression for High-Priced Laptops (Cambridge Computers)

This notebook answers **Problem 4 (a)–(g)**.

Problem 4 parts (c) and (e) require a comparison against the final **linear** regression model
from Problem 1, so that model is re-fit below in a short reference section before Problem 4 begins.

## Setup

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

train = pd.read_csv("laptop_train.csv")
test  = pd.read_csv("laptop_test.csv")

print("train:", train.shape, " test:", test.shape)
train.head()

train: (665, 9)  test: (280, 9)


,InventoryID,Company,TypeName,GPU,Screen,Memory,Weight,Rating,Price
0,6,Asus,Gaming,Nvidia,17.3,16,2.90,1,2122.0
1,15,Asus,Gaming,Nvidia,17.3,16,2.73,3,2049.9
2,17,Asus,Gaming,Nvidia,15.6,16,2.50,4,1799.0
3,18,Asus,Gaming,Nvidia,17.3,16,4.00,10,998.0
4,38,Asus,Gaming,Nvidia,15.6,8,2.30,6,1649.0


---
## Problem 1 reference model (needed for parts 4c and 4e)

Final linear model from Problem 1: all of `Company`, `TypeName`, `GPU`, `Screen`, `Memory`,
`Weight` and `Rating` are retained. `InventoryID` is excluded — it is an internal stocking
identifier with no managerial meaning (and it is insignificant, p ≈ 0.41). Every remaining
variable is significant at the 5% level (as a group, for the categorical variables), so
backward elimination removes nothing further.

In [2]:
lin_formula = "Price ~ C(Company) + C(TypeName) + C(GPU) + Screen + Memory + Weight + Rating"
lin_model = smf.ols(lin_formula, data=train).fit()
print(lin_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.660
Model:                            OLS   Adj. R-squared:                  0.654
Method:                 Least Squares   F-statistic:                     115.1
Date:                Tue, 22 Sep 2026   Prob (F-statistic):          8.75e-145
Time:                        16:50:19   Log-Likelihood:                -4809.1
No. Observations:                 665   AIC:                             9642.
Df Residuals:                     653   BIC:                             9696.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

---
## Problem 4 setup: create the binary outcome `high`

`high = 1` if `Price >= 500` Euros, `high = 0` otherwise, in **both** train and test.

In [3]:
train["high"] = (train["Price"] >= 500).astype(int)
test["high"]  = (test["Price"]  >= 500).astype(int)

print("Train high counts:\n", train["high"].value_counts().sort_index())
print("\nTest high counts:\n",  test["high"].value_counts().sort_index())
print("\nProportion high -- train: %.4f | test: %.4f" % (train["high"].mean(), test["high"].mean()))

Train high counts:
 high
0    115
1    550
Name: count, dtype: int64

Test high counts:
 high
0     53
1    227
Name: count, dtype: int64

Proportion high -- train: 0.8271 | test: 0.8107


---
## Problem 4(a)

Logistic regression predicting `high` from all independent variables in Table 1
(`InventoryID` is an ID, not a predictor). **No variable selection is performed.**

In [4]:
log_formula = "high ~ C(Company) + C(TypeName) + C(GPU) + Screen + Memory + Weight + Rating"
log_model = smf.logit(log_formula, data=train).fit()
print(log_model.summary())

         Current function value: 0.251696
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                   high   No. Observations:                  665
Model:                          Logit   Df Residuals:                      653
Method:                           MLE   Df Model:                           11
Date:                Tue, 22 Sep 2026   Pseudo R-squ.:                  0.4534
Time:                        16:50:19   Log-Likelihood:                -167.38
converged:                      False   LL-Null:                       -306.24
Covariance Type:            nonrobust   LLR p-value:                 4.281e-53
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   29.4861   5959.094      0.005      0.996   -1.17e+04    1.17e+04
C(Company)[T.Dell]     

/Users/chrislowzx/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


**Answer (a).** The fitted logistic regression is shown above (pseudo-R² = 0.453, LLR p-value ≈ 4.3e-53).

*One caveat to note:* every one of the 99 `Gaming` laptops in the training set is high-priced, so
`TypeName` produces **quasi-complete separation**. This is why statsmodels reports a convergence
warning and why the `Intercept` and the two `TypeName` coefficients have enormous standard errors —
those individual coefficients are not identified. All other coefficients, and all fitted
probabilities, are still well behaved and usable.

In [5]:
# Evidence of the separation: Gaming is perfectly predicted
pd.crosstab(train["TypeName"], train["high"])

high,0,1
TypeName,,
Gaming,0,99
Notebook,114,347
Ultrabook,1,104


---
## Problem 4(b)

Which variables are significant in predicting the probability of a high price?

In [6]:
sig = pd.DataFrame({
    "coef":   log_model.params,
    "p_value": log_model.pvalues,
})
sig["signif_5pct"] = np.where(sig["p_value"] < 0.05, "YES", "no")
print(sig.round(4).to_string())

print("\nSignificant at 5%:", [v for v in sig.index[sig["p_value"] < 0.05] if v != "Intercept"])

                             coef  p_value signif_5pct
Intercept                 29.4861   0.9961          no
C(Company)[T.Dell]         1.2456   0.0129         YES
C(Company)[T.HP]           1.4910   0.0013         YES
C(Company)[T.Lenovo]      -0.0525   0.9061          no
C(TypeName)[T.Notebook]  -17.6031   0.9976          no
C(TypeName)[T.Ultrabook] -16.1946   0.9978          no
C(GPU)[T.Intel]           -0.1053   0.7661          no
C(GPU)[T.Nvidia]           2.0148   0.0009         YES
Screen                    -1.1954   0.0001         YES
Memory                     0.7421   0.0000         YES
Weight                     1.5361   0.0889          no
Rating                    -0.0904   0.0660          no

Significant at 5%: ['C(Company)[T.Dell]', 'C(Company)[T.HP]', 'C(GPU)[T.Nvidia]', 'Screen', 'Memory']


**Answer (b).** At the 5% level the significant predictors are **`Memory`** (p < 0.001),
**`Screen`** (p < 0.001), **`GPU = Nvidia`** (p = 0.001), and **`Company = HP`** (p = 0.001) and
**`Company = Dell`** (p = 0.013) relative to the Asus baseline. `Weight` (p = 0.089) and `Rating`
(p = 0.066) are significant only at the 10% level, and `GPU = Intel` and `Company = Lenovo` are not
significant. The `TypeName` p-values (≈ 0.998) are meaningless here because of the separation
described in part (a).

---
## Problem 4(c)

Compare the significant variables of the logistic model with those of the Problem 1 linear model.

In [7]:
compare = pd.DataFrame({
    "linear_coef":  lin_model.params,
    "linear_p":     lin_model.pvalues,
    "logit_coef":   log_model.params,
    "logit_p":      log_model.pvalues,
})
compare["linear_sig_5pct"] = np.where(compare["linear_p"] < 0.05, "YES", "no")
compare["logit_sig_5pct"]  = np.where(compare["logit_p"]  < 0.05, "YES", "no")
compare["same_significance"] = np.where(
    compare["linear_sig_5pct"] == compare["logit_sig_5pct"], "same", "DIFFERENT")
print(compare.round(4).to_string())

                          linear_coef  linear_p  logit_coef  logit_p linear_sig_5pct logit_sig_5pct same_significance
Intercept                   1434.2466    0.0000     29.4861   0.9961             YES             no         DIFFERENT
C(Company)[T.Dell]            86.7254    0.0438      1.2456   0.0129             YES            YES              same
C(Company)[T.HP]             170.3954    0.0001      1.4910   0.0013             YES            YES              same
C(Company)[T.Lenovo]          35.3279    0.4020     -0.0525   0.9061              no             no              same
C(TypeName)[T.Notebook]      -76.2954    0.1943    -17.6031   0.9976              no             no              same
C(TypeName)[T.Ultrabook]     416.3939    0.0000    -16.1946   0.9978             YES             no         DIFFERENT
C(GPU)[T.Intel]              165.4297    0.0000     -0.1053   0.7661             YES             no         DIFFERENT
C(GPU)[T.Nvidia]             218.1386    0.0000      2.0

**Answer (c).** They largely agree but are not identical. `Memory`, `Screen`, `GPU = Nvidia`,
`Company = Dell` and `Company = HP` are significant in **both** models, and `Company = Lenovo` is
insignificant in both. The models differ on `GPU = Intel`, `Weight` and `Rating`, which are
significant at 5% in the linear model but not in the logistic model, and on `TypeName = Ultrabook`,
significant in the linear model but not estimable in the logistic model due to separation. This is
expected: the binary outcome discards the magnitude of price, so variables that shift price within
the "high" range lose explanatory power.

---
## Problem 4(d)

For each significant variable, does an increase in its value raise or lower the probability of being
high-priced?

In [8]:
signif = compare.loc[(compare["logit_p"] < 0.05) & (compare.index != "Intercept")].copy()
signif["direction"] = np.where(signif["logit_coef"] > 0,
                               "INCREASES P(high)", "DECREASES P(high)")
signif["odds_ratio"] = np.exp(signif["logit_coef"])
print(signif[["logit_coef", "odds_ratio", "logit_p", "direction"]].round(4).to_string())

                    logit_coef  odds_ratio  logit_p          direction
C(Company)[T.Dell]      1.2456      3.4750   0.0129  INCREASES P(high)
C(Company)[T.HP]        1.4910      4.4416   0.0013  INCREASES P(high)
C(GPU)[T.Nvidia]        2.0148      7.4991   0.0009  INCREASES P(high)
Screen                 -1.1954      0.3026   0.0001  DECREASES P(high)
Memory                  0.7421      2.1003   0.0000  INCREASES P(high)


**Answer (d).**

- **`Memory` (+0.742, OR ≈ 2.10):** each extra GB of RAM roughly doubles the odds of being
  high-priced. Sensible — RAM is a direct, costly component upgrade.
- **`Screen` (−1.195, OR ≈ 0.30):** a larger screen *lowers* the probability of a high price.
  Counter-intuitive on its own, but it reflects that, holding RAM/GPU/type fixed, big screens are
  characteristic of cheap bulk notebooks while premium ultrabooks are small; worth further
  investigation.
- **`GPU = Nvidia` (+2.015, OR ≈ 7.50):** a discrete Nvidia GPU raises the odds of a high price
  ~7.5× versus AMD. Very sensible — discrete gaming GPUs are expensive.
- **`Company = HP` (+1.491, OR ≈ 4.44) and `Company = Dell` (+1.246, OR ≈ 3.48):** both are more
  likely than Asus to be high-priced, consistent with brand positioning.

---
## Problem 4(e)

For which variables do the linear and logistic coefficients share a sign, and for which do they differ?

In [9]:
signs = compare[["linear_coef", "logit_coef"]].copy()
signs["linear_sign"] = np.sign(signs["linear_coef"]).map({1.0: "+", -1.0: "-", 0.0: "0"})
signs["logit_sign"]  = np.sign(signs["logit_coef"]).map({1.0: "+", -1.0: "-", 0.0: "0"})
signs["agreement"]   = np.where(signs["linear_sign"] == signs["logit_sign"], "SAME", "DIFFERENT")
print(signs.round(4).to_string())

                          linear_coef  logit_coef linear_sign logit_sign  agreement
Intercept                   1434.2466     29.4861           +          +       SAME
C(Company)[T.Dell]            86.7254      1.2456           +          +       SAME
C(Company)[T.HP]             170.3954      1.4910           +          +       SAME
C(Company)[T.Lenovo]          35.3279     -0.0525           +          -  DIFFERENT
C(TypeName)[T.Notebook]      -76.2954    -17.6031           -          -       SAME
C(TypeName)[T.Ultrabook]     416.3939    -16.1946           +          -  DIFFERENT
C(GPU)[T.Intel]              165.4297     -0.1053           +          -  DIFFERENT
C(GPU)[T.Nvidia]             218.1386      2.0148           +          +       SAME
Screen                      -120.9323     -1.1954           -          -       SAME
Memory                        87.3725      0.7421           +          +       SAME
Weight                       271.0986      1.5361           +          +    

**Answer (e).** The signs agree for **`Screen` (−), `Memory` (+), `Weight` (+), `Rating` (−),
`GPU = Nvidia` (+), `Company = Dell` (+) and `Company = HP` (+)** — the same directional story in
both models. They differ for **`GPU = Intel`** (+165 in the linear model, −0.105 in the logit) and
**`Company = Lenovo`** (+35 linear, −0.053 logit); in both cases the logistic coefficient is
statistically insignificant, so the sign flip is not meaningful. **`TypeName`** also flips sign for
`Ultrabook` (+416 linear, −16.19 logit), but its logistic coefficients are not identified because of
the separation noted in part (a).

---
## Problem 4(f)

Probability that a Lenovo Ultrabook with an Intel GPU, `Screen = 8`, `Memory = 8`, `Weight = 4.2`,
`Rating = 7` is high-priced.

The logistic model gives

$$P(\text{high}=1 \mid x) = \frac{1}{1 + e^{-z}}, \qquad
z = \beta_0 + \beta_{\text{Lenovo}} + \beta_{\text{Ultrabook}} + \beta_{\text{Intel}}
 + \beta_{\text{Screen}}(8) + \beta_{\text{Memory}}(8) + \beta_{\text{Weight}}(4.2)
 + \beta_{\text{Rating}}(7)$$

(the Dell/HP, Notebook and Nvidia indicators are all 0 for this laptop).

In [10]:
new_laptop = pd.DataFrame({
    "InventoryID": [4096],
    "Company":     ["Lenovo"],
    "TypeName":    ["Ultrabook"],
    "GPU":         ["Intel"],
    "Screen":      [8.0],
    "Memory":      [8],
    "Weight":      [4.2],
    "Rating":      [7],
})

b = log_model.params
terms = {
    "Intercept":              b["Intercept"],
    "Company=Lenovo":         b["C(Company)[T.Lenovo]"],
    "TypeName=Ultrabook":     b["C(TypeName)[T.Ultrabook]"],
    "GPU=Intel":              b["C(GPU)[T.Intel]"],
    "Screen x 8.0":           b["Screen"] * 8.0,
    "Memory x 8":             b["Memory"] * 8,
    "Weight x 4.2":           b["Weight"] * 4.2,
    "Rating x 7":             b["Rating"] * 7,
}
for k, v in terms.items():
    print(f"  {k:<22} = {v:>12.4f}")

z = sum(terms.values())
p_manual = 1 / (1 + np.exp(-z))
p_smf    = log_model.predict(new_laptop).iloc[0]

print(f"\n  z (log-odds)           = {z:.4f}")
print(f"  P(high=1) manual       = {p_manual:.8f}")
print(f"  P(high=1) via .predict = {p_smf:.8f}")
print(f"  1 - P(high=1)          = {1 - p_manual:.3e}")

  Intercept              =      29.4861
  Company=Lenovo         =      -0.0525
  TypeName=Ultrabook     =     -16.1946
  GPU=Intel              =      -0.1053
  Screen x 8.0           =      -9.5635
  Memory x 8             =       5.9365
  Weight x 4.2           =       6.4516
  Rating x 7             =      -0.6325

  z (log-odds)           = 15.3257
  P(high=1) manual       = 0.99999978
  P(high=1) via .predict = 0.99999978
  1 - P(high=1)          = 2.209e-07


**Answer (f).** Substituting the fitted coefficients gives a log-odds of
**z ≈ 15.33**, so

$$P(\text{high}=1) = \frac{1}{1+e^{-15.33}} \approx 0.99999978 \approx 1.00$$

The model predicts this laptop is essentially certain to be high-priced. Note that `Screen = 8`
inches lies far outside the training range (12.5–17.3 in), so this is an extrapolation and the
probability should be read with caution.

---
## Problem 4(g)

Apply the model to the test set with a 0.5 probability cutoff and compute accuracy.

In [11]:
test_probs = log_model.predict(test)
test_pred  = (test_probs >= 0.5).astype(int)

conf = pd.crosstab(test["high"], test_pred,
                   rownames=["Actual"], colnames=["Predicted"])
print("Confusion matrix (test set):")
print(conf, "\n")

accuracy = (test_pred == test["high"]).mean()
baseline = test["high"].mean()   # always predict "high"

print(f"Test accuracy (cutoff 0.5) = {accuracy:.4f}  ({(test_pred == test['high']).sum()} / {len(test)})")
print(f"Baseline accuracy          = {baseline:.4f}")

Confusion matrix (test set):
Predicted   0    1
Actual            
0          27   26
1          15  212 

Test accuracy (cutoff 0.5) = 0.8536  (239 / 280)
Baseline accuracy          = 0.8107


**Answer (g).** Using a 0.5 cutoff, the model classifies **239 of 280** test laptops correctly, for
an out-of-sample accuracy of **0.8536 (85.4%)**. This beats the naive baseline of always predicting
"high", which is correct 81.1% of the time.